# Certamen de laboratorio · Minería de Datos · Unidad II

**martes 27 de octubre de 2026** · **individual** · 120 minutos · 100 puntos · 40 % de la nota de práctica

**Nombre:** _escriba aquí su nombre_ · **RUT:** _escriba aquí su RUT_

## Antes de empezar (lea esto primero)

1. **Archivo → Guardar una copia en Drive.** Trabaje siempre en su copia.
2. Ejecute la **celda ▶ entregada** (la primera de código). Debe ver: *Datos cargados: 200 fichas (160 de entrenamiento y 40 de prueba), 300 pedidos, 40 semanas y 60 estudiantes.*
3. Cada tarea tiene siempre las mismas partes, en este orden:

| Ícono | Qué es | Qué hace usted |
|:-:|---|---|
| 📖 | **Contexto** | Entender el problema del negocio y para qué sirve la tarea |
| 💡 | **Concepto clave** | Repasar la idea que hay detrás de lo que va a programar |
| 🎯 | **Qué hacer** | Leer los pasos |
| 🧩 | **Función con su firma** | Completar **solo el cuerpo**. No cambie el nombre ni los parámetros. El docstring dice qué recibe, qué devuelve y trae un ejemplo |
| 🧪 | **Prueba** con las tablas del certamen de cátedra | Ejecutarla. Debe dar **exactamente** el «resultado esperado» (ya conoce esos números de la cátedra). Si no da, corrija la función antes de seguir |
| 📊 | **Análisis** con los datos del laboratorio | Escribir el código que pide la tarea |
| ✅ | **Resultado esperado** | Comparar: dice **qué debe mostrar** su celda 📊 y en qué formato |
| ✍️ | **Respuesta** | Responder con **sus propios números** |

4. **De los datos al conocimiento.** Cada tarea termina con una pregunta 🧠. El curso trabaja con una escalera de tres peldaños:

| Peldaño | Qué es | Ejemplo en la clínica |
|---|---|---|
| **Dato** | Un hecho suelto | «La ficha 1023 recibió correo, vive lejos y no asistió» |
| **Información** | Datos procesados que muestran un patrón | «El árbol acierta 9 de cada 10 fichas de prueba» |
| **Conocimiento** | Información convertida en una **decisión** | «Hay que llamar a quienes reciben correo y viven lejos» |

   Su código produce **información**; sus respuestas 🧠 deben llegar al **conocimiento**.

5. Puede usar **todos los recursos** (apuntes, guías, internet, IA). Pero se evalúa que **interprete sus resultados**: una respuesta genérica, sin sus números, no tiene puntaje.
6. Tiempo sugerido: **Parte I** hasta el minuto 40, **Parte II** hasta el 80 y **Parte III** hasta el 120.
7. Al terminar: **Archivo → Descargar → Descargar .ipynb**, y súbalo al **Aula Virtual** como `Apellido_Nombre.ipynb`.

| Parte | Caso | Tareas | Puntos |
|---|---|:-:|:-:|
| I | Clínica veterinaria: ZeroR, OneR, árbol, prueba y sobreajuste | 1 a 5 | 35 |
| II | Ferretería online: soporte, confianza, lift y regresión | 6 a 9 | 35 |
| III | Liceo: k-means por rondas y sus gráficas | 10 a 12 | 30 |

In [ ]:
# ▶ CELDA ENTREGADA — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/certamen-lab/"
clinica = pd.read_csv(RUTA + "lab_clinica.csv")              # Parte I
pedidos = pd.read_csv(RUTA + "lab_pedidos.csv")              # Parte II (Apriori)
semanas = pd.read_csv(RUTA + "lab_semanas.csv")              # Parte II (regresión)
liceo = pd.read_csv(RUTA + "lab_liceo.csv")                  # Parte III
cat_clinica = pd.read_csv(RUTA + "catedra_clinica.csv")      # tablas del certamen de cátedra,
cat_pedidos = pd.read_csv(RUTA + "catedra_pedidos.csv")      # solo para las pruebas 🧪
cat_liceo = pd.read_csv(RUTA + "catedra_liceo.csv")

entrenamiento = clinica[clinica["Conjunto"] == "entrenamiento"]
prueba = clinica[clinica["Conjunto"] == "prueba"]
canastas = pedidos.groupby("Pedido")["Producto"].apply(set).tolist()          # un conjunto por pedido
cat_canastas = cat_pedidos.groupby("Pedido")["Producto"].apply(set).tolist()
puntos = list(zip(liceo["Estudio"], liceo["Actividad"]))                     # pares (x, y)
cat_puntos = list(zip(cat_liceo["Estudio"], cat_liceo["Actividad"]))

ATRIBUTOS = ["Especie", "Recordatorio", "Distancia", "Edad"]
CAT_ATRIBUTOS = ["Especie", "Recordatorio", "Distancia"]
PRODUCTOS = ['Pintura', 'Brocha', 'Rodillo', 'Lija', 'Guantes', 'Huincha']
ESTRELLAS_INICIO = [(1, 9), (2, 1), (3, 3)]

print(f"Datos cargados: {len(clinica)} fichas ({len(entrenamiento)} de entrenamiento y {len(prueba)} de prueba), "
      f"{len(canastas)} pedidos, {len(semanas)} semanas y {len(puntos)} estudiantes.")

---
# Parte I · Clínica veterinaria (35 puntos)

La clínica quiere predecir **qué dueños no traerán a su mascota al control de vacunas**. La tabla `clinica` tiene **200 fichas** con:

| Columna | Valores |
|---|---|
| `Ficha` | número de ficha (1001 a 1200) |
| `Especie` | Perro · Gato |
| `Recordatorio` | WhatsApp · Correo · Ninguno |
| `Distancia` | Cerca · Lejos |
| `Edad` | Cachorro · Adulto · Senior |
| `Asistio` | **Sí · No** ← la clase que se quiere predecir |
| `Conjunto` | `entrenamiento` (160 fichas, para **construir** los modelos) · `prueba` (40 fichas, para **evaluarlos**) |

Ya están separadas en las variables **`entrenamiento`** y **`prueba`**. Los modelos se construyen **solo** con `entrenamiento`.

## Tarea 1 · ZeroR: la línea base (4 puntos)

### 📖 Contexto

Antes de construir cualquier modelo, la clínica necesita saber **cuánto se acierta «sin pensar»**. Si la mayoría de los dueños asiste, predecir «todos asisten» ya acierta bastante; un modelo solo vale la pena si **mejora** eso.

### 💡 Concepto clave

> **ZeroR** («cero reglas») no mira ningún atributo: predice siempre la **clase mayoritaria**. No sirve para decidir nada, pero fija la **línea base**, el acierto mínimo que cualquier modelo debe superar. Un **error** es una ficha donde la predicción no coincide con lo que pasó; el **acierto** es aciertos / total.

### 🎯 Qué hacer

1. Complete `zeror(df, clase)` y `acierto(reales, predichas)` (lea el docstring de cada una: dice qué reciben, qué devuelven y trae un ejemplo).
2. Ejecute la celda 🧪 de prueba y compare con el resultado esperado.
3. En la celda 📊: aplique `zeror()` a `entrenamiento` y calcule el **acierto en `prueba`**. Para eso, ZeroR predice la clase mayoritaria de entrenamiento para **todas** las fichas de prueba.

In [ ]:
# 🧩 Tarea 1 · complete el cuerpo de la(s) función(es)
def zeror(df, clase):
    """ZeroR: predice SIEMPRE la clase que más se repite.

    Recibe:   df    → la tabla (DataFrame)
              clase → el nombre de la columna que se quiere predecir, aquí "Asistio"
    Devuelve: (clase_mayoritaria, errores)
              errores = cuántas filas NO son de la clase mayoritaria (ahí ZeroR se equivoca)
    Ejemplo:  zeror(cat_clinica, "Asistio")  →  ("Sí", 6)
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass


def acierto(reales, predichas):
    """Proporción de aciertos: cuántas predicciones coinciden con lo real, dividido por el total.

    Recibe:   dos listas (o columnas) del mismo largo
    Devuelve: un número entre 0 y 1
    Ejemplo:  acierto(["Sí", "No", "Sí"], ["Sí", "Sí", "Sí"])  →  0.6667   (2 de 3)
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 1 con las tablas del certamen de cátedra: ejecútela sin cambiarla
c, e = zeror(cat_clinica, "Asistio")
print(c, e)
# ✅ Debe mostrar: Sí 6
print(round(acierto(["Sí", "No", "Sí"], ["Sí", "Sí", "Sí"]), 4))
# ✅ Debe mostrar: 0.6667

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`Sí 6`**: en las 14 fichas de la cátedra hay 8 «Sí» y 6 «No»: ZeroR predice «Sí» y se equivoca en las 6 «No». Si le da 8, está contando aciertos, no errores.
- **`0.6667`**: 2 de las 3 predicciones coinciden con lo real: 2/3 = 0,6667.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 1 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** Tres datos: **la clase que predice ZeroR** («Sí» o «No»), **sus errores en entrenamiento** (un número entero entre 0 y 160) y **su acierto en prueba** (un porcentaje).

### ✍️ Responda

**1. Sus resultados:** ¿Qué predice ZeroR, cuántos errores comete en entrenamiento y qué porcentaje acierta en prueba? ¿Para qué le sirve este número en las tareas siguientes?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si la clínica no usara ningún modelo y simplemente supusiera que todos vienen, ¿en cuántas de cada 10 mascotas se equivocaría?

_Su respuesta:_

## Tarea 2 · OneR y el mejor atributo (8 puntos)

### 📖 Contexto

La administradora quiere saber **qué dato de la ficha** (especie, recordatorio, distancia o edad) explica mejor si el dueño trae a su mascota.

### 💡 Concepto clave

> **OneR** («una regla») prueba cada atributo **por separado**. Para cada valor del atributo predice la clase más frecuente en ese valor, y se queda con el atributo que comete **menos errores**. Un atributo es útil si **cambia la predicción** según su valor; si en todos sus valores la mayoría es la misma, su regla dice lo mismo que ZeroR y no aporta.

### 🎯 Qué hacer

1. Complete `oner(df, atributo, clase)` y `mejor_atributo(df, atributos, clase)`.
2. Ejecute la prueba 🧪.
3. En 📊: arme una tabla con los **errores de OneR para cada uno de los 4 atributos** (`ATRIBUTOS`) en `entrenamiento`, indique el ganador y muestre sus reglas.

In [ ]:
# 🧩 Tarea 2 · complete el cuerpo de la(s) función(es)
def oner(df, atributo, clase):
    """OneR con UN atributo: una regla por cada valor del atributo.

    Para cada valor del atributo: la regla es la clase que más se repite en ese valor, y los
    errores son las filas de ese valor que tienen la otra clase. Se suman los errores.
    Recibe:   df, atributo (p. ej. "Recordatorio"), clase ("Asistio")
    Devuelve: (reglas, errores)
              reglas = diccionario {valor: clase predicha}
    Ejemplo:  oner(cat_clinica, "Recordatorio", "Asistio")
              →  ({"Correo": "Sí", "Ninguno": "No", "WhatsApp": "Sí"}, 2)
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass


def mejor_atributo(df, atributos, clase):
    """Aplica oner() a cada atributo de la lista y devuelve el que comete MENOS errores.

    Recibe:   df, atributos (lista de nombres de columnas), clase
    Devuelve: (atributo_ganador, reglas_del_ganador, errores_del_ganador)
              Si dos atributos empatan, gana el que aparece primero en la lista.
    Ejemplo:  mejor_atributo(cat_clinica, CAT_ATRIBUTOS, "Asistio")  →  ("Recordatorio", {...}, 2)
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 2 con las tablas del certamen de cátedra: ejecútela sin cambiarla
reglas, e = oner(cat_clinica, "Recordatorio", "Asistio")
print(dict(sorted(reglas.items())), e)
# ✅ Debe mostrar: {'Correo': 'Sí', 'Ninguno': 'No', 'WhatsApp': 'Sí'} 2
atributo, reglas, e = mejor_atributo(cat_clinica, CAT_ATRIBUTOS, "Asistio")
print(atributo, e)
# ✅ Debe mostrar: Recordatorio 2

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`{'Correo': 'Sí', 'Ninguno': 'No', 'WhatsApp': 'Sí'} 2`**: con Correo hay 3 Sí y 2 No (regla Sí, 2 errores); con Ninguno, 0 Sí y 4 No (regla No, 0 errores); con WhatsApp, 5 Sí y 0 No (regla Sí, 0 errores). Total: 2 errores.
- **`Recordatorio 2`**: Especie comete 6 errores, Recordatorio 2 y Distancia 4: gana el de menos errores.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 2 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** Una **tabla de 4 filas** (un atributo por fila) con sus **errores en entrenamiento**, el **atributo ganador** y **su regla** (una clase para cada valor del atributo).

### ✍️ Responda

**1. Sus resultados:** ¿Qué atributo gana y cuál es su regla, en palabras? ¿Hay algún atributo que **no le gane a ZeroR**? Explique por qué, **mirando sus reglas**.

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Con la regla ganadora: ¿qué le recomendaría a la clínica sobre **cómo enviar los recordatorios**?

_Su respuesta:_

## Tarea 3 · Árbol de decisión de dos niveles (10 puntos)

### 📖 Contexto

OneR usa un solo dato. Pero puede que el correo funcione **solo** con quienes viven cerca. Un **árbol** combina datos: primero una pregunta y, en el grupo donde todavía hay mezcla, una segunda.

### 💡 Concepto clave

> Un **árbol de decisión** es un diagrama de preguntas. La **raíz** es la pregunta que mejor separa. Una **rama pura** tiene una sola clase y no necesita más preguntas. En una **rama impura** se busca otra pregunta usando **solo las fichas de esa rama**. Cada camino de la raíz a una **hoja** es una regla «si … y … entonces …». Más preguntas no siempre ayudan: solo se agrega una si **reduce los errores**.

### 🎯 Qué hacer

1. Complete `arbol_dos_niveles(df, atributos, clase)` siguiendo **los 4 pasos** del docstring, y `predecir(arbol, fila)`.
2. Ejecute la prueba 🧪 (las 14 fichas de la cátedra).
3. En 📊: construya el árbol con `entrenamiento` y `ATRIBUTOS`, **muéstrelo** y cuente sus **errores en entrenamiento** (cuántas fichas predice mal).

In [ ]:
# 🧩 Tarea 3 · complete el cuerpo de la(s) función(es)
def arbol_dos_niveles(df, atributos, clase):
    """Árbol de decisión de DOS preguntas, construido con la idea de OneR.

    Pasos:
      1. La RAÍZ es el mejor atributo (mejor_atributo con todos los atributos).
      2. Para cada valor de la raíz se toman SOLO las filas de esa rama.
      3. Si la rama tiene errores (no es pura), se busca el mejor atributo entre los que quedan,
         usando SOLO las filas de esa rama.
      4. Esa segunda pregunta se usa solo si comete MENOS errores que dejar la rama como hoja
         (es decir, que predecir la clase mayoritaria de la rama). Si no mejora, queda como hoja.
    Devuelve un diccionario así (ejemplo con las 14 fichas de la cátedra):
      {"raiz": "Recordatorio",
       "ramas": {"Correo":   {"pregunta": "Distancia", "reglas": {"Cerca": "Sí", "Lejos": "No"}, "clase": "Sí"},
                 "Ninguno":  {"pregunta": None, "reglas": {}, "clase": "No"},
                 "WhatsApp": {"pregunta": None, "reglas": {}, "clase": "Sí"}}}
      "clase" es la clase mayoritaria de la rama (se usa si la rama es hoja o si llega un valor nuevo).
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass


def predecir(arbol, fila):
    """Clasifica UNA fila siguiendo el árbol.

    Se mira el valor de la raíz en la fila → se entra a esa rama. Si la rama tiene segunda
    pregunta, se mira ese atributo y se usa su regla; si no (o si el valor es nuevo), se devuelve
    la "clase" de la rama.
    Recibe:   arbol (lo que devuelve arbol_dos_niveles) y fila (una fila del DataFrame)
    Devuelve: "Sí" o "No"
    Uso:      [predecir(arbol, fila) for _, fila in prueba.iterrows()]
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 3 con las tablas del certamen de cátedra: ejecútela sin cambiarla
arbol_cat = arbol_dos_niveles(cat_clinica, CAT_ATRIBUTOS, "Asistio")
print(arbol_cat["raiz"], arbol_cat["ramas"]["Correo"]["pregunta"], arbol_cat["ramas"]["WhatsApp"]["pregunta"])
# ✅ Debe mostrar: Recordatorio Distancia None
errores = sum(predecir(arbol_cat, fila) != fila["Asistio"] for _, fila in cat_clinica.iterrows())
print(errores)
# ✅ Debe mostrar: 0

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`Recordatorio Distancia None`**: la raíz es Recordatorio (el ganador de OneR); la rama Correo es impura y Distancia la separa sin errores; la rama WhatsApp ya es pura, así que no lleva segunda pregunta.
- **`0`**: con la segunda pregunta en la rama Correo, el árbol clasifica bien las 14 fichas.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 3 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** El **árbol**: la raíz y una línea por cada rama (con su segunda pregunta o como hoja), y **sus errores en entrenamiento** (un número entero, menor o igual que los de OneR).

### ✍️ Responda

**1. Sus resultados:** Escriba las reglas de su árbol **en palabras**. ¿Cuántos errores comete en entrenamiento, comparado con OneR? ¿Hay ramas que quedaron como **hoja aunque tienen errores**? ¿Por qué el algoritmo no les agregó una segunda pregunta?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Traduzca su árbol a un **protocolo para la recepcionista**: ¿a qué dueños hay que llamar por teléfono antes del control?

_Su respuesta:_

## Tarea 4 · ¿Cuál funciona mejor con fichas nuevas? (7 puntos)

### 📖 Contexto

Los modelos se construyen con fichas del pasado, pero se usarán con mascotas **futuras**. Las 40 fichas de **prueba** simulan ese futuro: ningún modelo las vio.

### 💡 Concepto clave

> **Entrenamiento** son los datos con que se construye el modelo; **prueba**, datos nuevos con que se evalúa. Medir con los mismos datos de entrenamiento es como corregir una prueba con las respuestas a la vista: el resultado siempre se ve mejor de lo que es. **El acierto que importa es el de prueba.**

### 🎯 Qué hacer

1. No hay función nueva: use las que ya programó.
2. En 📊: calcule el **acierto en `prueba`** (las 40 fichas que ningún modelo vio) de **ZeroR, OneR (con el ganador) y su árbol**, y muéstrelos en una tabla.

In [ ]:
# 📊 Tarea 4 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** Una **tabla de 3 filas** (ZeroR, OneR y árbol) con el **acierto en prueba** de cada uno (entre 0 y 1, o en %).

### ✍️ Responda

**1. Sus resultados:** ¿Qué modelo recomendaría a la clínica y por qué? ¿Por qué el resultado en prueba es distinto del de entrenamiento, y por qué es el que importa?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si la clínica aplica el modelo recomendado a **100 mascotas** el próximo mes, ¿en cuántas esperaría acertar? ¿Es una garantía?

_Su respuesta:_

## Tarea 5 · El número de ficha (6 puntos)

### 📖 Contexto

Un practicante propone usar el **número de ficha** como atributo, porque «da cero errores».

### 💡 Concepto clave

> **Sobreajuste**: el modelo **memoriza** los datos de entrenamiento en vez de aprender un patrón que se repita. Pasa con atributos que **identifican** a cada caso (RUT, número de ficha, patente): cada valor aparece una sola vez, así que la «regla» es perfecta en entrenamiento e inútil con casos nuevos.

### 🎯 Qué hacer

1. En 📊: aplique `oner()` con el atributo **`"Ficha"`** a `entrenamiento` y muestre sus errores.
2. Luego use esas reglas para predecir las fichas de `prueba` (con `reglas.get(ficha)`: devuelve `None` si la ficha no tiene regla) y calcule el acierto.

In [ ]:
# 📊 Tarea 5 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** **Dos números**: los errores con Ficha en entrenamiento y el acierto en prueba. Las predicciones de prueba aparecen como **`None`**: no existe regla para esas fichas.

### ✍️ Responda

**1. Sus resultados:** ¿Qué pasa en entrenamiento y qué pasa en prueba? ¿Cómo se llama este problema? ¿Por qué el número de ficha no debe usarse como atributo?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** ¿Qué le diría al practicante? Dé **otro ejemplo** de un atributo que tendría el mismo problema en otro negocio.

_Su respuesta:_

---
# Parte II · Ferretería online (35 puntos)

**Apriori.** La tabla `pedidos` tiene una fila por producto comprado (`Pedido`, `Producto`). Ya está convertida en **`canastas`**: una lista de **300 conjuntos**, uno por pedido. Por ejemplo, `{'Pintura', 'Brocha', 'Guantes'}`. Los 6 productos están en `PRODUCTOS`.

📄 **Antes de la tarea 6, lea la guía de apoyo «soporte(), confianza() y lift()».**

**Regresión.** La tabla `semanas` tiene 40 semanas con `Avisos` (gasto en avisos, en miles de $), `Cyber` (1 si hubo evento Cyber esa semana, 0 si no) y `Pedidos` (pedidos de esa semana).

## Tarea 6 · Soporte de cada producto (6 puntos)

### 📖 Contexto

La ferretería online quiere agregar la sección **«Otros clientes también llevaron…»**. Primero debe saber qué productos se venden lo suficiente para que valga la pena analizarlos.

### 💡 Concepto clave

> El **soporte** es la proporción de pedidos que contienen un producto (o un grupo de productos). Apriori fija un **soporte mínimo** y descarta lo que no llega: si un producto es poco frecuente, ningún par con él puede ser frecuente. Así evita analizar combinaciones que casi nunca ocurren.

### 🎯 Qué hacer

1. Lea la **guía de apoyo** «soporte(), confianza() y lift()» (PDF).
2. Complete `soporte(canastas, productos)`.
3. Ejecute la prueba 🧪.
4. En 📊: calcule el soporte de **cada uno de los 6 productos** (`PRODUCTOS`) en los 300 pedidos, muéstrelo en una tabla y marque cuáles llegan al **soporte mínimo de 10 %**.

In [ ]:
# 🧩 Tarea 6 · complete el cuerpo de la(s) función(es)
def soporte(canastas, productos):
    """Soporte: en qué proporción de TODOS los pedidos aparecen juntos los productos dados.

    Recibe:   canastas  → lista de pedidos; cada pedido es un conjunto, p. ej. {"Pintura", "Guantes"}
              productos → lista con uno o más productos, p. ej. ["Pintura"] o ["Pintura", "Brocha"]
    Devuelve: un número entre 0 y 1
    Ejemplo:  soporte(cat_canastas, ["Pintura"])            →  0.5   (5 de 10 pedidos)
              soporte(cat_canastas, ["Pintura", "Brocha"])  →  0.4   (4 de 10 pedidos)
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 6 con las tablas del certamen de cátedra: ejecútela sin cambiarla
print(soporte(cat_canastas, ["Pintura"]), soporte(cat_canastas, ["Pintura", "Brocha"]))
# ✅ Debe mostrar: 0.5 0.4

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`0.5 0.4`**: Pintura está en 5 de los 10 pedidos de la cátedra (0,5); Pintura y Brocha juntas, en 4 de 10 (0,4).

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 6 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** Una **tabla de 6 filas**: el producto, su **soporte** (entre 0 y 1, o en %) y si es **frecuente** (soporte ≥ 10 %).

### ✍️ Responda

**1. Sus resultados:** ¿Qué productos no son frecuentes? ¿Qué hace Apriori con ellos al buscar pares, y por qué tiene sentido?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** ¿Qué producto se lleva en **casi todos** los pedidos? ¿Qué problema podría causar eso al buscar reglas? (Lo comprobará en la tarea 8.)

_Su respuesta:_

## Tarea 7 · Confianza y lift de tres reglas (8 puntos)

### 📖 Contexto

Una regla **A → B** dice «quien lleva A también lleva B». La ferretería quiere saber si esas reglas reflejan una **relación real** o son casualidad.

### 💡 Concepto clave

> La **confianza (A → B)** es la proporción de los pedidos con A que también tienen B; se divide por los pedidos con A. El **lift** compara esa confianza con la popularidad de B. **Lift > 1**: llevar A aumenta la probabilidad de llevar B (relación real). **Lift ≈ 1**: no hay relación. **Lift < 1**: se compran juntos menos de lo esperado.

### 🎯 Qué hacer

1. Complete `confianza(canastas, a, b)` y `lift(canastas, a, b)` (pueden usar su `soporte()`).
2. Ejecute la prueba 🧪.
3. En 📊: calcule **soporte, confianza y lift** de **R1: Pintura → Brocha**, **R2: Brocha → Pintura** y **R3: Lija → Guantes** en los 300 pedidos, en una tabla.

In [ ]:
# 🧩 Tarea 7 · complete el cuerpo de la(s) función(es)
def confianza(canastas, a, b):
    """Confianza de la regla a → b: de los pedidos que tienen a, qué proporción tiene también b.

    Confianza(a → b) = soporte(a y b) / soporte(a)      (se divide por el producto de la IZQUIERDA)
    Ejemplo:  confianza(cat_canastas, "Pintura", "Brocha")  →  0.8   (4 de los 5 pedidos con Pintura)
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass


def lift(canastas, a, b):
    """Lift de la regla a → b: cuántas veces más se lleva b cuando se lleva a, comparado con el azar.

    Lift(a → b) = confianza(a → b) / soporte(b)
    Lectura:  mayor que 1 → asociación real · igual a 1 → no hay relación · menor que 1 → se evitan
    Ejemplo:  lift(cat_canastas, "Pintura", "Brocha")  →  2.0
              lift(cat_canastas, "Lija", "Guantes")    →  1.0
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 7 con las tablas del certamen de cátedra: ejecútela sin cambiarla
print(round(confianza(cat_canastas, "Pintura", "Brocha"), 4), round(lift(cat_canastas, "Pintura", "Brocha"), 4), round(lift(cat_canastas, "Lija", "Guantes"), 4))
# ✅ Debe mostrar: 0.8 2.0 1.0

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`0.8 2.0 1.0`**: confianza 4/5 = 0,8 (de 5 pedidos con Pintura, 4 llevan Brocha); lift 0,8/0,4 = 2; Lija → Guantes: confianza 0,8 y Guantes está en el 80 % de los pedidos → lift 1 (sin relación). Son los números del certamen de cátedra.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 7 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** Una **tabla de 3 filas** (R1, R2, R3) con **soporte, confianza y lift**. El lift es un número (por ejemplo 1,25), no un porcentaje.

### ✍️ Responda

**1. Sus resultados:** Interprete el **lift** de cada regla en una frase. ¿Por qué R1 y R2 tienen **distinta confianza** pero el **mismo lift**?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si un cliente pone **Pintura** en el carro, ¿conviene mostrarle Brocha? ¿Y si pone **Lija**, conviene mostrarle Guantes? Justifique con el lift.

_Su respuesta:_

## Tarea 8 · Todas las reglas: ¿cuál conviene recomendar? (7 puntos)

### 📖 Contexto

Con 6 productos hay **30 reglas posibles**. Revisarlas a mano no escala; por eso Apriori las recorre todas, filtra por soporte y las ordena.

### 💡 Concepto clave

> Ordenar por **confianza** favorece a los productos populares, porque casi cualquier regla que termine en ellos tiene confianza alta. Ordenar por **lift** muestra las asociaciones reales. La mejor regla para recomendar tiene **soporte suficiente** y un **lift claramente mayor que 1**.

### 🎯 Qué hacer

1. Complete `todas_las_reglas(canastas, productos, soporte_minimo)`.
2. Ejecute la prueba 🧪.
3. En 📊: obtenga todas las reglas de los 300 pedidos con **soporte mínimo 10 %**, muestre las **5 primeras** (ordenadas por lift) y, aparte, las **3 de mayor confianza**.

In [ ]:
# 🧩 Tarea 8 · complete el cuerpo de la(s) función(es)
def todas_las_reglas(canastas, productos, soporte_minimo):
    """Revisa TODAS las reglas a → b (a distinto de b) y se queda con las que cumplen el soporte mínimo.

    Para cada par de productos distintos (en los dos sentidos: a → b y b → a), si
    soporte([a, b]) >= soporte_minimo, guarda una fila con: antecedente, consecuente,
    soporte, confianza y lift.
    Devuelve: un DataFrame con esas 5 columnas, ORDENADO por lift de mayor a menor.
    Ejemplo:  todas_las_reglas(cat_canastas, PRODUCTOS, 0.3)  →  8 reglas; las dos primeras con lift 2.0
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 8 con las tablas del certamen de cátedra: ejecútela sin cambiarla
tabla_cat = todas_las_reglas(cat_canastas, PRODUCTOS, 0.3)
print(len(tabla_cat), round(tabla_cat["lift"].iloc[0], 4))
# ✅ Debe mostrar: 8 2.0

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`8 2.0`**: con los 10 pedidos de la cátedra hay 8 reglas con soporte ≥ 30 %, y la de mayor lift tiene lift 2 (Pintura ↔ Brocha).

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 8 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** **Dos tablas**: las 5 reglas de mayor lift y las 3 de mayor confianza, cada una con antecedente, consecuente, soporte, confianza y lift.

### ✍️ Responda

**1. Sus resultados:** ¿Qué regla recomendaría para la sección «Otros clientes también llevaron» y por qué? ¿La regla de **mayor confianza** es la mejor? Explíquelo con sus números.

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Escriba la **recomendación final para el gerente**: qué productos mostrar juntos y por qué no basta con mirar la confianza.

_Su respuesta:_

## Tarea 9 · Regresión: ¿cuántos pedidos tendremos? (14 puntos)

### 📖 Contexto

La ferretería quiere **planificar el stock y los despachos** según cuánto invierta en avisos y si hay evento Cyber.

### 💡 Concepto clave

> La **regresión lineal** ajusta y = a + b₁·x₁ + b₂·x₂. Cada **coeficiente** dice cuánto cambia y cuando esa variable sube en 1 y las demás quedan fijas. Una variable 0/1 como Cyber suma su coeficiente solo cuando vale 1. El **R²** dice qué proporción de la variación de y explica el modelo. Un modelo solo vale dentro del **rango de datos** con que se ajustó; usarlo fuera de ese rango es **extrapolar**.

### 🎯 Qué hacer

1. No hay función que completar: escriba el código en 📊.
2. **X** = las columnas `Avisos` y `Cyber` de `semanas`; **y** = la columna `Pedidos`.
3. Ajuste un `LinearRegression()` y muestre: el **intercepto** (`modelo.intercept_`), los **dos coeficientes** (`modelo.coef_`) y el **R²** (`modelo.score(X, y)`).
4. Prediga los pedidos de una semana con **$60 mil** en avisos, **sin** Cyber y **con** Cyber (`modelo.predict(...)` recibe una tabla con las mismas columnas que X).
5. Muestre el **mínimo y el máximo** de `Avisos` en los datos.

In [ ]:
# 📊 Tarea 9 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** **Cinco números del modelo** (intercepto, coeficiente de Avisos, coeficiente de Cyber, R² entre 0 y 1) más **dos predicciones** (sin y con Cyber) y el **rango de Avisos** observado.

### ✍️ Responda

**1. Sus resultados:** (a) Explique al dueño, en una frase cada uno, qué significan el intercepto y los dos coeficientes. (b) ¿Cuántos pedidos se esperan con $60 mil sin y con Cyber? (c) El dueño quiere gastar **$500 mil** en avisos y usar el modelo para saber cuántos pedidos tendrá: ¿por qué **no** sirve? (d) ¿Qué significa su R²? ¿Es el porcentaje de semanas que el modelo acierta?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** El dueño duda entre gastar **$10 mil más en avisos** en una semana normal o **concentrar la campaña en la semana del Cyber**. ¿Qué le dicen sus coeficientes? ¿Qué le recomendaría antes de pensar en invertir $500 mil?

_Su respuesta:_

---
# Parte III · Liceo (30 puntos)

La tabla `liceo` tiene **60 estudiantes** con `Estudio` (horas de estudio a la semana) y `Actividad` (horas de actividad física a la semana). Ya están en **`puntos`**: una lista de pares `(x, y)`.

Se buscan **3 grupos** con k-means, partiendo de las estrellas **`ESTRELLAS_INICIO` = (1; 9), (2; 1) y (3; 3)**.

**Recuerde:** una **ronda** = **(1) elegir:** cada estudiante va a la estrella más cercana + **(2) moverse:** cada estrella va al promedio de su grupo. Se repite hasta que en una ronda **nadie cambie de grupo**.

## Tarea 10 · Elegir y moverse: asignar() y actualizar() (8 puntos)

### 📖 Contexto

La orientadora del liceo quiere formar **grupos de estudiantes parecidos** para ofrecer talleres distintos, sin tener categorías definidas de antemano.

### 💡 Concepto clave

> **k-means** agrupa **sin etiquetas** (aprendizaje no supervisado). Cada grupo tiene una **estrella** (su centro). Una **ronda** tiene dos pasos: **elegir** (cada estudiante va a la estrella más cercana; se compara d² = (x − x★)² + (y − y★)², sin sacar raíz) y **moverse** (cada estrella va al promedio de su grupo).

### 🎯 Qué hacer

1. Complete `asignar(puntos, estrellas)` (paso 1 de cada ronda) y `actualizar(puntos, grupos, estrellas)` (paso 2).
2. Ejecute la prueba 🧪 con los 6 estudiantes de la cátedra.

In [ ]:
# 🧩 Tarea 10 · complete el cuerpo de la(s) función(es)
def asignar(puntos, estrellas):
    """Paso 1 de una ronda (ELEGIR): cada punto elige la estrella más cercana.

    Recibe:   puntos    → lista de pares (x, y), p. ej. [(4, 9), (7, 4), ...]
              estrellas → lista de pares (x, y), una por grupo
    Devuelve: lista de grupos, uno por punto: 0 = primera estrella, 1 = segunda, ...
    Para comparar basta d² = (x − x★)² + (y − y★)², sin raíz.
    Ejemplo:  asignar(cat_puntos, [(7, 10), (2, 2)])  →  [0, 1, 0, 1, 1, 1]
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass


def actualizar(puntos, grupos, estrellas):
    """Paso 2 de una ronda (MOVERSE): cada estrella se va al promedio de los puntos de su grupo.

    Recibe:   puntos, grupos (lo que devuelve asignar) y las estrellas actuales
    Devuelve: lista de nuevas estrellas (x, y). Si un grupo quedara vacío, su estrella no se mueve.
    Ejemplo:  actualizar(cat_puntos, [0, 1, 0, 1, 1, 1], [(7, 10), (2, 2)])  →  [(7.0, 7.0), (2.5, 2.5)]
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 10 con las tablas del certamen de cátedra: ejecútela sin cambiarla
print(asignar(cat_puntos, [(7, 10), (2, 2)]))
# ✅ Debe mostrar: [0, 1, 0, 1, 1, 1]
nuevas = actualizar(cat_puntos, [0, 1, 0, 1, 1, 1], [(7, 10), (2, 2)])
print([(round(float(x), 2), round(float(y), 2)) for x, y in nuevas])
# ✅ Debe mostrar: [(7.0, 7.0), (2.5, 2.5)]

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`[0, 1, 0, 1, 1, 1]`**: es la ronda 1 de la cátedra: Amanda y Camila eligen ★1 (grupo 0); Bruno, Dante, Elisa y Franco eligen ★2 (grupo 1).
- **`[(7.0, 7.0), (2.5, 2.5)]`**: ★1 = promedio de Amanda y Camila = (7; 7); ★2 = promedio de los otros cuatro = (2,5; 2,5).

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

## Tarea 11 · El ciclo de rondas (12 puntos)

### 📖 Contexto

Una sola ronda no basta: al moverse las estrellas, algunos estudiantes quedan más cerca de otra y **cambian de grupo**.

### 💡 Concepto clave

> El algoritmo repite rondas hasta que **nadie cambia de grupo**: ahí las estrellas ya no se mueven y los grupos son estables. Un estudiante puede cambiar de grupo **sin que cambien sus datos**, porque lo que se movió fue la estrella.

### 🎯 Qué hacer

1. Complete `kmeans_por_rondas(puntos, estrellas_inicio, graficar=True)` siguiendo los pasos del docstring. **Se detiene cuando en una ronda nadie cambia de grupo.**
2. Ejecute la prueba 🧪 (sin gráficas).
3. En 📊: córrala con los **60 estudiantes** (`puntos`) desde `ESTRELLAS_INICIO`, **todavía con `graficar=False`**, y muestre una tabla con, por ronda: **número de ronda, cuántos cambiaron de grupo y dónde estaban las estrellas**. Al final muestre las estrellas finales y cuántos estudiantes quedaron en cada grupo.

In [ ]:
# 🧩 Tarea 11 · complete el cuerpo de la(s) función(es)
def kmeans_por_rondas(puntos, estrellas_inicio, graficar=True):
    """El CICLO de k-means: repite rondas (elegir + moverse) hasta que en una ronda NADIE cambie.

    En cada ronda:
      1. grupos = asignar(puntos, estrellas)
      2. se cuenta cuántos puntos cambiaron de grupo respecto de la ronda anterior
         (en la ronda 1 no hay con qué comparar: cambios = None)
      3. se guarda la ronda en el historial y, si graficar=True, se grafica con graficar_ronda()
      4. si cambios == 0 → se termina; si no → estrellas = actualizar(...) y se sigue
    Devuelve: (estrellas_finales, grupos_finales, historial)
              historial = lista con un diccionario por ronda: {"ronda": 1, "cambios": None, "estrellas": [...]}
    Ejemplo:  con cat_puntos y [(7, 10), (2, 2)] → 3 rondas, cambios [None, 1, 0]
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 11 con las tablas del certamen de cátedra: ejecútela sin cambiarla
estrellas_cat, grupos_cat, historial_cat = kmeans_por_rondas(cat_puntos, [(7, 10), (2, 2)], graficar=False)
print(len(historial_cat), [h["cambios"] for h in historial_cat])
print([(round(float(x), 2), round(float(y), 2)) for x, y in estrellas_cat])
# ✅ Debe mostrar: 3 [None, 1, 0]  /  [(7.0, 6.0), (1.0, 2.0)]

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`3 [None, 1, 0] / [(7.0, 6.0), (1.0, 2.0)]`**: igual que en la cátedra: 3 rondas; en la ronda 1 no hay con qué comparar (None), en la 2 cambia Bruno (1) y en la 3 nadie (0) → termina. Estrellas finales (7; 6) y (1; 2).

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 11 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** Una **tabla con una fila por ronda** (ronda, cambios, estrellas). La última ronda debe tener **0 cambios**. Debajo: las **3 estrellas finales** y **cuántos estudiantes** hay en cada grupo (suman 60).

### ✍️ Responda

**1. Sus resultados:** ¿En cuántas rondas termina? ¿Cuántos estudiantes cambian de grupo en cada ronda? ¿Dónde quedan las estrellas y cuántos estudiantes tiene cada grupo?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Mirando las **estrellas finales**: ¿qué tipo de estudiante representa cada grupo? Póngales nombre y proponga **un taller** para uno de ellos.

_Su respuesta:_

## Tarea 12 · Las gráficas de cada ronda (10 puntos)

### 📖 Contexto

Una tabla de coordenadas cuesta leerla; una **gráfica** muestra de un vistazo cómo se forman los grupos.

### 💡 Concepto clave

> **Representar el conocimiento**: la misma información se entiende mejor con la forma adecuada. En cada gráfica los colores son los grupos y la ★ es la estrella; comparar rondas sucesivas muestra cómo se mueven.

### 🎯 Qué hacer

1. Complete `graficar_ronda(puntos, grupos, estrellas, titulo)` cumpliendo **todos los requisitos** del docstring.
2. En 📊: vuelva a correr `kmeans_por_rondas(puntos, ESTRELLAS_INICIO, graficar=True)`. Debe aparecer **una gráfica por ronda**.

In [ ]:
# 🧩 Tarea 12 · complete el cuerpo de la(s) función(es)
def graficar_ronda(puntos, grupos, estrellas, titulo):
    """Dibuja UNA ronda: los puntos coloreados según su grupo y cada estrella como ★ grande.

    Requisitos de la gráfica:
      • cada grupo con un color distinto, y su estrella del MISMO color, con marcador "*" grande
      • ejes de 0 a 10, con nombre: "Horas de estudio" (x) y "Horas de actividad física" (y)
      • el título que se recibe (debe decir el número de ronda)
    No devuelve nada: solo muestra la gráfica (plt.show()).
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 📊 Tarea 12 · escriba aquí su análisis con los datos del laboratorio

**✅ Resultado esperado de su análisis 📊:** **Una gráfica por ronda**: los 60 puntos coloreados por grupo, cada estrella como ★ grande del color de su grupo, ejes de 0 a 10 con nombre y un título con el número de ronda.

### ✍️ Responda

**1. Sus resultados:** Describa en 2 o 3 líneas lo que muestran las gráficas: ¿cómo se mueven las estrellas de una ronda a la siguiente y por qué se detiene el algoritmo?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si tuviera que mostrarle **una sola gráfica** a la orientadora, ¿cuál elegiría y por qué?

_Su respuesta:_

---
## ✔️ Para entregar

1. **Entorno de ejecución → Reiniciar y ejecutar todo**: revise que todo corra sin errores.
2. **Archivo → Descargar → Descargar .ipynb**.
3. Súbalo al **Aula Virtual** como `Apellido_Nombre.ipynb`.